# Public Matches Model (Hero-Decay Only)

A lightweight training notebook that:
- loads high-MMR public matches from the DB,
- computes hero half-life win-rate features only,
- trains a simple classifier, and
- saves a BentoML-compatible model tagged as `dota_oracle_pub_match_model`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from types import SimpleNamespace
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

from dota_oracle_common.utils.set_logging import get_logger
from dota_oracle_common.postgresql import DatabaseManager
from dota_oracle_common.repositories.public_match_repository import PublicMatchRepository
from dota_oracle_common.models.inference.schema import PerformanceMetrics, VersionMetaData, ModelMetaData

from dota_oracle_pipeline.feature_engineering.batch.hero_wr_features.decay import (
    HeroWinrateDecayFeatureGenerator,
)

from model_factory.save_model import save_sklearn_model

logger = get_logger(__name__)
logger.info("Notebook initialized")

2025-11-04 09:55:36,350 - __main__ - INFO - Notebook initialized


## 1) Load public matches (DB)

In [3]:
async def load_public_matches(limit: int = 20000, start_time: int | None = None, end_time: int | None = None):
    session_factory = DatabaseManager.get_session_factory()
    async with session_factory() as session:
        repo = PublicMatchRepository(session)
        matches = await repo.get_public_matches(limit=limit, start_time=start_time, end_time=end_time)
        # commit read-only session to release pool resources
        await session.commit()
        return sorted(matches, key=lambda m: m.start_time)

public_matches = await load_public_matches(limit=80000)
len(public_matches)

2025-11-04 09:55:54,334 - dota_oracle_common.postgresql - INFO - Creating database engine with pool_size=10 and max_overflow=5
2025-11-04 09:55:54,386 - dota_oracle_common.postgresql - INFO - Successfully initialized database for 'dota2' at localhost
2025-11-04 09:55:56,580 - dota_oracle_common.repositories.base_repository - INFO - Retrieved 80000 records for PublicMatchTable
2025-11-04 09:55:56,581 - dota_oracle_common.repositories.public_match_repository - INFO - Found 80000 PublicMatchTable rows.


80000

## 2) Adapt to Match-like objects (for generator)

In [4]:
def to_match_like(m):
    # Wrap PublicMatchTable into a shape expected by feature generators:
    # - has match_id, start_time, slot_*_hero_id
    # - has .outcome.radiant_win boolean
    outcome = SimpleNamespace(radiant_win=bool(m.radiant_win))
    fields = {
        'match_id': m.match_id,
        'start_time': m.start_time,
        'outcome': outcome,
        # hero ids (radiant 0..4, dire 128..132)
        'slot_0_hero_id': m.slot_0_hero_id,
        'slot_1_hero_id': m.slot_1_hero_id,
        'slot_2_hero_id': m.slot_2_hero_id,
        'slot_3_hero_id': m.slot_3_hero_id,
        'slot_4_hero_id': m.slot_4_hero_id,
        'slot_128_hero_id': m.slot_128_hero_id,
        'slot_129_hero_id': m.slot_129_hero_id,
        'slot_130_hero_id': m.slot_130_hero_id,
        'slot_131_hero_id': m.slot_131_hero_id,
        'slot_132_hero_id': m.slot_132_hero_id,
    }
    return SimpleNamespace(**fields)

match_like = [to_match_like(m) for m in public_matches]
len(match_like)

80000

## 3) Generate hero half-life features (wide) and aggregate to a single diff

In [5]:
HERO_PRIOR_MEAN = 0.5
HERO_PRIOR_COUNT = 50
HERO_HALF_LIFE_DAYS = 45  # keep in sync with pro defaults for comparability

gen = HeroWinrateDecayFeatureGenerator()
hero_wide = gen.generate_wide_format(
    match_like, prior_mean=HERO_PRIOR_MEAN, prior_count=HERO_PRIOR_COUNT, half_life_days=HERO_HALF_LIFE_DAYS
)
df_hero = pd.DataFrame([r.model_dump() for r in hero_wide])

# Aggregate: avg radiant and dire hero winrates, then their difference
rad_cols = [f'hero_{i}_win_rate' for i in range(5)]
dir_cols = [f'hero_{i}_win_rate' for i in range(128, 133)]
df_hero['radiant_avg_hero_wr'] = df_hero[rad_cols].mean(axis=1)
df_hero['dire_avg_hero_wr'] = df_hero[dir_cols].mean(axis=1)
df_hero['radiant_dire_hero_wr_diff'] = df_hero['radiant_avg_hero_wr'] - df_hero['dire_avg_hero_wr']

df_y = pd.DataFrame({
    'match_id': [m.match_id for m in public_matches],
    'radiant_win': [bool(m.radiant_win) for m in public_matches],
})

dataset = df_hero[['match_id', 'radiant_dire_hero_wr_diff']].merge(df_y, on='match_id', how='inner')
dataset.head()

,match_id,radiant_dire_hero_wr_diff,radiant_win
0,8297854801,0.000000e+00,False
1,8297856003,0.000000e+00,False
2,8297856315,-2.193337e-08,False
3,8297856917,-3.921542e-03,False
4,8297857114,-1.161380e-02,False


## 4) Train a simple classifier

In [6]:
X = dataset[['radiant_dire_hero_wr_diff']].values.astype(np.float64)
y = dataset['radiant_win'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
ll = log_loss(y_test, y_prob)
acc, auc, ll

(0.5475625, 0.5571491755410782, 0.6852499582285214)

## 5) Save model to BentoML store

In [7]:
model_name = 'dota_oracle_pub_match_model'
feature_columns = ['radiant_dire_hero_wr_diff']

perf = PerformanceMetrics(accuracy=float(acc), roc_auc=float(auc), log_loss=float(ll))
ver_meta = VersionMetaData(performance_metrics=perf, feature_columns=feature_columns)

metadata = ModelMetaData(
    name=model_name,
    description='Public matches model using hero-decay-only features',
    intended_use='Quick, lightweight predictions for public matches only',
    limitations='limited signal strength, data primarily from ancient and above matches, might not generalize to all skill brackets',
    version='1.0',
    trained_date=datetime.now(),
    version_metadata=ver_meta,
)

save_sklearn_model(model=clf, model_name=model_name, metadata=metadata)

2025-11-04 10:01:23,069 - model_factory.save_model - INFO - attempting to save model dota_oracle_pub_match_model
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
2025-11-04 10:01:24,160 - model_factory.save_model - INFO - Model saved: Model(tag="dota_oracle_pub_match_model:hbsl3ufzmwu2lryn")
